# AM5061 · Week 7 · Counter-flow heat exchanger

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


---
## The case

Liquid cooling for a **50 kW GPU rack**. You are building a counter-flow heat
exchanger **from first principles**: not one lumped UA, but N segments in
series with a metal wall between the streams.

Deliverable **D-7**: run N = 2, 5, 10, 20, 40 at UA = 400, 2000 and 6000 W/K,
and tabulate the numerical effectiveness against the analytical one.

### The question worth answering

How many segments do you need for 1% error, and **why does that number grow as
NTU rises?** That is the real deliverable. The rest is arithmetic.


## 1. The discretised exchanger

Each segment is a well-mixed volume, so its outlet temperature *is* its bulk
temperature. Hot and cold run in opposite directions; the wall couples them
segment by segment with conductance `2·UA/N` on each side (two in series gives
`UA/N` per segment, and N of those gives UA).


In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import fsolve
am.style_plots()

cp, rho = 4181.0, 995.6
T_HOT_IN, T_COLD_IN = 350.0, 290.0      # K
K_TOT, DP = 1e5, 2.0e4                  # loss coefficient and the driving dp

def hx(N=5, UA=400.0, Th_in=T_HOT_IN, Tc_in=T_COLD_IN, k_tot=K_TOT, dp=DP):
    """Steady state of an N-segment counter-flow exchanger."""
    m = np.sqrt(dp / k_tot)             # both sides, same resistance
    C_ = m * cp
    G  = 2 * UA / N                     # hot->wall, and wall->cold

    def residual(v):
        Th, Tw, Tc = v[:N], v[N:2*N], v[2*N:]
        r = []
        for i in range(N):                       # hot marches 0 -> N-1
            th_in = Th_in if i == 0 else Th[i-1]
            r.append(C_*(th_in - Th[i]) - G*(Th[i] - Tw[i]))
        for i in range(N):                       # wall: what arrives, leaves
            r.append(G*(Th[i] - Tw[i]) - G*(Tw[i] - Tc[i]))
        for i in range(N):                       # cold marches N-1 -> 0
            tc_in = Tc_in if i == N-1 else Tc[i+1]
            r.append(C_*(Tc[i] - tc_in) - G*(Tw[i] - Tc[i]))
        return np.array(r)

    guess = np.concatenate([np.full(N, 340.), np.full(N, 320.), np.full(N, 300.)])
    sol = fsolve(residual, guess)
    Th, Tw, Tc = sol[:N], sol[N:2*N], sol[2*N:]
    NTU = UA / (m * cp)
    return {"N": N, "UA, W/K": UA, "m_dot, kg/s": m, "NTU": NTU,
            "eps_numerical": (Th_in - Th[-1])/(Th_in - Tc_in),
            "eps_analytical": NTU/(1 + NTU),
            "T_hot_out, K": Th[-1], "T_cold_out, K": Tc[0],
            "_profiles": (Th, Tw, Tc)}

r = hx()
for k, v in r.items():
    if not k.startswith("_"):
        print(f"  {k:16s} {v:12.6f}" if isinstance(v, float) else f"  {k:16s} {v:12d}")


> **Check.** At N = 5, UA = 400 the numerical effectiveness is
> **0.170227**, against an analytical 0.176227.
>
> Both streams have the same `m·cp`, so the capacity ratio C_r = 1 and the
> counter-flow result collapses to **ε = NTU/(1+NTU)**. That is the only case
> where the analytical answer is this simple, which is why the case is built
> on it.


## 2. Grid convergence

The actual deliverable.

In [ ]:
Ns  = [2, 5, 10, 20, 40, 80, 160]
UAs = [400.0, 2000.0, 6000.0]

rows = []
for UA in UAs:
    for N in Ns:
        q = hx(N=N, UA=UA)
        rows.append({"N": N, "UA, W/K": UA, "NTU": q["NTU"],
                     "eps_numerical": q["eps_numerical"],
                     "eps_analytical": q["eps_analytical"],
                     "error, %": 100*(q["eps_numerical"]-q["eps_analytical"])/q["eps_analytical"]})

print(f"{'UA':>7}{'NTU':>8}{'N':>6}{'eps_num':>11}{'eps_an':>10}{'err %':>9}")
for row in rows:
    print(f"{row['UA, W/K']:7.0f}{row['NTU']:8.3f}{row['N']:6d}"
          f"{row['eps_numerical']:11.6f}{row['eps_analytical']:10.6f}{row['error, %']:9.2f}")


In [ ]:
fig, ax = plt.subplots()
for UA in UAs:
    sub = [r_ for r_ in rows if r_["UA, W/K"] == UA]
    ax.plot([s["N"] for s in sub], [abs(s["error, %"]) for s in sub], "o-",
            label=f"UA = {UA:.0f} W/K   (NTU = {sub[0]['NTU']:.2f})")
ax.axhline(1.0, color=am.MUTED, ls="--", lw=1.4)
ax.text(2.2, 1.15, "1% error", color=am.MUTED, fontsize=9)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("segments  N"); ax.set_ylabel("|error| vs analytical  (%)")
ax.set_title("More NTU needs more segments for the same accuracy")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("Segments needed for under 1% error:")
for UA in UAs:
    sub = [r_ for r_ in rows if r_["UA, W/K"] == UA]
    ok = [s["N"] for s in sub if abs(s["error, %"]) < 1.0]
    print(f"  UA {UA:6.0f} (NTU {sub[0]['NTU']:5.2f}) -> "
          f"{ok[0] if ok else '>160'} segments")


**Why the number grows with NTU.** Each well-mixed segment is
isothermal, so a chain of N of them approximates a smooth exponential
temperature profile by a staircase of N steps. At low NTU the true profile is
nearly straight and a coarse staircase fits it well. At high NTU the profile
curves hard near the inlet, and you need more steps to follow the curvature.
Resolution has to track the gradient, not the length.


## 3. The temperature profile

See the staircase for yourself.

In [ ]:
fig, ax = plt.subplots()
for N, style in ((5, "o-"), (40, "-")):
    Th, Tw, Tc = hx(N=N, UA=2000.)["_profiles"]
    xpos = (np.arange(N) + 0.5) / N
    ax.plot(xpos, Th, style, color=am.ORANGE, lw=2, ms=6,
            label=f"hot, N={N}", alpha=1.0 if N == 5 else 0.55)
    ax.plot(xpos, Tc, style, color=am.BLUE, lw=2, ms=6,
            label=f"cold, N={N}", alpha=1.0 if N == 5 else 0.55)
ax.set_xlabel("position along the exchanger, hot inlet at 0")
ax.set_ylabel("temperature  (K)")
ax.set_title("Counter-flow profiles at UA = 2000 W/K")
ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()


## 4. The deliverable

In [ ]:
path = am.to_excel("AM5061_D7_HeatExchanger.xlsx",
    {"Grid convergence": rows},
    title="AM5061 D-7 . Counter-flow HX, grid convergence",
    summary=[("Hot inlet", T_HOT_IN, "K"), ("Cold inlet", T_COLD_IN, "K"),
             ("Mass flow each side", r["m_dot, kg/s"], "kg/s"),
             ("Capacity ratio C_r", 1.0, "-"),
             ("eps at N=5, UA=400", r["eps_numerical"], "-"),
             ("eps analytical at UA=400", r["eps_analytical"], "-")],
    sources=[("Water properties", "rho 995.6 kg/m3, cp 4181 J/kgK"),
             ("Analytical effectiveness", "counter-flow, C_r = 1: eps = NTU/(1+NTU)"),
             ("Discretisation", "N well-mixed segments per side, wall capacity between")])
print("written:", path)


## What to hand in

1. The convergence table, all three UA values.
2. The error plot.
3. The number of segments for 1% error at each NTU, **and one paragraph on why
   it grows**.
4. The workbook.

A note on cost: this is a nonlinear solve in 3N unknowns. At N = 160 that is
480 equations, and it still runs in under a second. Discretisation is cheap.
Not knowing how much you need is what is expensive.
